# GPT2 Speech Generation 

## 1. Setup & paths

In [5]:
MODEL_DIR = "../outputs/gpt2/final"
print("Model dir:", MODEL_DIR)

Model dir: ../outputs/gpt2/final


## 2. Define the model + generation functions



In [ ]:
from functools import lru_cache

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel


def pick_device() -> torch.device:
    """Prefer CUDA, then Apple MPS, then CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@lru_cache(maxsize=2)
def load_model(model_dir: str = MODEL_DIR, device: str | None = None):
    """Load (and cache) the fine-tuned GPT-2 LM-head model and tokenizer.

    Cached on (model_dir, device): the first call loads weights from disk; later
    calls with the same args return the same warm objects.

    Returns (tokenizer, model, device).
    """
    dev = torch.device(device) if device else pick_device()
    print(f"Loading GPT-2 from {model_dir} onto {dev}")

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    tokenizer.pad_token = tokenizer.eos_token

    model = GPT2LMHeadModel.from_pretrained(model_dir)
    model.config.pad_token_id = tokenizer.eos_token_id
    model.eval()
    model.to(dev)

    return tokenizer, model, dev


def generate_speech(
    prompt: str,
    *,
    model_dir: str = MODEL_DIR,
    device: str | None = None,
    max_new_tokens: int = 200,
    temperature: float = 0.9,
    top_p: float = 0.95,
    top_k: int = 50,
    repetition_penalty: float = 1.2,
    no_repeat_ngram_size: int = 3,
    num_return_sequences: int = 1,
    seed: int | None = None,

) -> list[str]:
    """Generate speech text conditioned on ``prompt`` via nucleus sampling.

    Args:
        prompt: Opening text to condition on, e.g. "Mr. Speaker, I rise today".
        max_new_tokens: How many tokens to generate beyond the prompt.
        temperature: Sampling sharpness. Higher = more random/creative.
        top_p: Nucleus sampling cutoff (cumulative probability mass).
        top_k: Cap on candidate tokens per step (0 disables).
        repetition_penalty: >1.0 discourages repeating tokens (GPT-2 loops a lot).
        no_repeat_ngram_size: Forbid repeating any n-gram of this size.
        num_return_sequences: How many independent samples to return.
        seed: If set, makes generation reproducible.

    Returns:
        List of generated strings (each includes the prompt).
    """
    if not prompt or not prompt.strip():
        raise ValueError("prompt must be a non-empty string")

    tokenizer, model, dev = load_model(model_dir, device)

    if seed is not None:
        torch.manual_seed(seed)
        if dev.type == "cuda":
            torch.cuda.manual_seed_all(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(dev)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id,
        )

    return [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]

## 3. Load (warm) the model once

In [7]:
tokenizer, model, device = load_model(MODEL_DIR)
print(f"Loaded onto {device} — {sum(p.numel() for p in model.parameters()):,} parameters")

Loading GPT-2 from ../outputs/gpt2/final onto mps


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded onto mps — 354,823,168 parameters


## 4. Single generation

Edit `PROMPT` and re-run. `seed` is set for reproducibility while exploring —
remove it for fresh samples each run.

In [8]:
PROMPT = "Mr. Speaker, I rise today"

speeches = generate_speech(
    PROMPT,
    max_new_tokens=200,
    temperature=0.9,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    seed=42,
)

print(speeches[0])

Loading GPT-2 from ../outputs/gpt2/final onto mps


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Mr. Speaker, I rise today to congratulate the Town of St. Francis Park on being named the Outstanding Community in the State of Minnesota in 2016 by the Minnesota Council for Higher Education. The town was created in 1972 by the Central New York Railroad Company, located just outside of Rochester, MN. The railroad's former patrons include local businesses that would go on and prosper under its continued operation, including a grocery store as well as restaurants, hardware stores, an ice cream parlor at St. Franciscan Church, and a sports arena. St. Franciks is also home community to a number of community-based organizations, such as the St. Frank Church and the United Way of St Frank in addition to the local elementary school, St. Andrew's Catholic School in East End, and many other churches around the city. Throughout this time, Mr. Speaker and my colleagues from across Minnesota have worked tirelessly with the community to make sure they are able to continue to thrive. The impact of 

## 5. Temperature sweep

Same prompt and seed across temperatures to feel how sampling sharpness changes
the output. Lower = safer/repetitive, higher = more creative/chaotic.

In [9]:
PROMPT = "Mr. Speaker, I rise today"

for temp in (0.5, 0.7, 0.9, 1.1):
    out = generate_speech(PROMPT, max_new_tokens=120, temperature=temp, seed=42)[0]
    print(f"\n{'=' * 70}\ntemperature = {temp}\n{'=' * 70}")
    print(out)


temperature = 0.5
Mr. Speaker, I rise today to recognize the service of a great American and a good friend, John C. ``Jack'' Caffey, who has served as president for the past 2 years. Jack was born in New York City on November 11, 1954. He graduated from the University of Pennsylvania with a degree in English. In 1968 he joined the United States Army in Vietnam where his unit was assigned to the 3rd Battalion, 8th Infantry Regiment, 82nd Airborne Division. Upon returning home, Jack attended the University at Buffalo Law School. While attending law school, Jack earned his Master's Degree in Public Administration from

temperature = 0.7
Mr. Speaker, I rise today to recognize the service of David W. Moore, Jr., who recently passed away on June 4th at the age of 94 years old. Mr. Moore was a founding member and past president-elect of the American Legion, an outstanding military leader in his community, and a decorated veteran from World War II. He served honorably during the Korean war as

## 6. Sampling exploration (top_p / top_k / multiple samples)

Draw several independent samples from the same prompt to gauge variety, and try a
tighter nucleus (`top_p`) for more focused output.

In [15]:
PROMPT = "The American people deserve"

samples = generate_speech(
    PROMPT,
    model_dir="gpt2-medium",
    max_new_tokens=120,
    temperature=0.9,
    top_p=0.9,
    top_k=40,
    num_return_sequences=3
)

for i, s in enumerate(samples, 1):
    print(f"\n----- sample {i}/{len(samples)} -----")
    print(s)

Loading GPT-2 from gpt2-medium onto mps


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]


----- sample 1/3 -----
The American people deserve to know what is in the intelligence community's reports. The truth should matter."

----- sample 2/3 -----
The American people deserve to know how the U.S.-led coalition is using drones, and why they're killing innocent civilians," she said in a statement issued later Tuesday.
: The CIA declined comment for this story about its drone strike program on an alleged al Qaeda terrorist compound outside Yemen last month when one of their planes crashed into it during "targeted attack." As recently as April 2016 — before Trump took office — Obama told reporters that he didn't have any details related at all but would tell them if anything came out.: Last summer, ABC News first revealed another US military official who had served with SEAL Team

----- sample 3/3 -----
The American people deserve to know what happened."


, President Trump's campaign manager was fired after he admitted that the candidate had "inadvertently" used a private emai

In [16]:
PROMPT = "The American people deserve"

samples = generate_speech(
    PROMPT,
    max_new_tokens=120,
    temperature=0.9,
    top_p=0.9,
    top_k=40,
    num_return_sequences=3
)

for i, s in enumerate(samples, 1):
    print(f"\n----- sample {i}/{len(samples)} -----")
    print(s)


----- sample 1/3 -----
The American people deserve a full explanation of this situation. I would like to ask my distinguished friend from Massachusetts how much money we are talking about here in connection with the proposed loan program? I refer particularly especially now- and then onlyto our friends on that side who have been so vociferous against any type or kindof foreign aid program at allor what they call an international economicaid program whatsoever for Europe todayand why has it not worked out well under these conditions already without such programs as outlined by some Senators and Members of Congress during recent yearsI am sure they realize there will be no more loans than ever before

----- sample 2/3 -----
The American people deserve a real debate on the health care bill. We are in this mess because of the Republican refusal to work with Democrats and moderate Republicans on something that will bring real reform forward for all Americans so we can improve our economy a